# Occlusion — entraînement aligné sur la distribution TEST :
1. **Poids d'importance** vers la distribution test : `(1/30+gt) · p_test(gt)/p_train(gt)` (au lieu de `/p` qui visait l'uniforme) → estimateur non-biaisé de la métrique sur le test.
2. **Métrique de val rééchantillonnée** (`metric_fn_test`) pour la sélection du modèle 

In [1]:
import os, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
from scipy.ndimage import gaussian_filter1d
import torch, torch.nn as nn
import torchvision.transforms as T, torchvision.models as models
from torch.utils.data import Dataset, DataLoader

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type=="cuda" else "CPU")

BACKBONES=["convnext_small"]
IMG_SIZE=224
BATCH={"convnext_tiny":64,"convnext_small":48,"convnext_base":32,"swin_s":32}
EPOCHS=22
FREEZE_EPOCHS=2
LR_HEAD=1e-3; LR_BACKBONE=4e-5; WEIGHT_DECAY=5e-2
EMA_DECAY=0.999; GRAD_CLIP=1.0
VAL_FRAC=0.20; NUM_WORKERS=4; SEED=42

GENDER_POWER=0.5
WEIGHT_BINS=100; WEIGHT_SIGMA=2.0; OMEGA_CLIP=10.0
FAIRNESS_LAMBDA=0.0

OUT_DIR="/kaggle/working"; MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
torch.manual_seed(SEED); np.random.seed(SEED)

# Densite cible = distribution d occlusion du TEST (proxy: blend 0.00100).
# Remplace par le vrai histogramme test si tu l as (EDA du rapport).
TARGET_DENSITY=np.array([0.0352734, 0.0346087, 0.0333437, 0.0317853, 0.0304164, 0.0295609, 0.0292144, 0.0292156, 0.0294692, 0.0299305, 0.0304531, 0.0307956, 0.0308325, 0.0306825, 0.0305649, 0.0305754, 0.0306526, 0.0307157, 0.0307427, 0.0307038, 0.0304937, 0.0300095, 0.0292944, 0.0285313, 0.0278469, 0.0271467, 0.0261921, 0.0248172, 0.0230362, 0.0209932, 0.0188706, 0.0168114, 0.0148588, 0.0129644, 0.0110714, 0.0091909, 0.0074008, 0.0057916, 0.0044183, 0.0032878, 0.0023765, 0.0016586, 0.001118, 0.0007413, 0.0005012, 0.0003543, 0.0002559, 0.0001779, 0.0001123, 6.2e-05, 2.97e-05, 1.33e-05, 7.6e-06, 6.8e-06, 6.8e-06, 5.9e-06, 4e-06, 2.2e-06, 9e-07, 3e-07, 1e-07, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],dtype=float)
TARGET_DENSITY/=TARGET_DENSITY.sum()
assert len(TARGET_DENSITY)==WEIGHT_BINS, "TARGET_DENSITY = WEIGHT_BINS valeurs"
print("Backbones:",BACKBONES,"| EPOCHS:",EPOCHS,"| cible len:",len(TARGET_DENSITY))

Device: cuda | Tesla T4
Backbones: ['convnext_small'] | EPOCHS: 22 | cible len: 100


In [2]:
INPUT="/kaggle/input"; base=INPUT.rstrip("/").count("/")
TRAIN_CSV=TEST_CSV=None
for dp,dns,fns in os.walk(INPUT):
    if dp.count("/")-base>=5: dns[:]=[]
    if TRAIN_CSV is None and "train.csv" in fns: TRAIN_CSV=os.path.join(dp,"train.csv")
    if TEST_CSV  is None and "test_students.csv" in fns: TEST_CSV=os.path.join(dp,"test_students.csv")
assert TRAIN_CSV and TEST_CSV,"CSV introuvables"
sample_rel=pd.read_csv(TRAIN_CSV,nrows=1)["filename"].iloc[0]
IMAGE_DIR=None
for dp,dns,_ in os.walk(INPUT):
    if dp.count("/")-base>=5: dns[:]=[]
    if os.path.exists(os.path.join(dp,sample_rel)): IMAGE_DIR=dp; break
assert IMAGE_DIR,"Images introuvables"
print("IMAGE_DIR:",IMAGE_DIR)

IMAGE_DIR: /kaggle/input/datasets/mouhamedsamb2001/data-challenge-dataset/crops/Crop_224_5fp_100K


In [3]:
from sklearn.model_selection import train_test_split
from torch.utils.data import WeightedRandomSampler
df_all=pd.read_csv(TRAIN_CSV).dropna().reset_index(drop=True)
df_test=pd.read_csv(TEST_CSV).dropna().reset_index(drop=True)
df_train,df_val=train_test_split(df_all,test_size=VAL_FRAC,stratify=df_all["gender"],random_state=SEED)
df_train=df_train.reset_index(drop=True); df_val=df_val.reset_index(drop=True)
yv,gv=df_val["FaceOcclusion"].values, df_val["gender"].values
print("Train",len(df_train),"Val",len(df_val))

_edges=np.linspace(0,1,WEIGHT_BINS+1)
def _bin(g): return np.clip(np.digitize(g,_edges)-1,0,WEIGHT_BINS-1)
_h=np.bincount(_bin(df_train["FaceOcclusion"].values),minlength=WEIGHT_BINS).astype(float)
_h=np.maximum(gaussian_filter1d(_h,WEIGHT_SIGMA),1e-6); p_train=_h/_h.sum()
def importance_ratio(g): return TARGET_DENSITY[_bin(g)]/(p_train[_bin(g)]+1e-6)

# (1) poids de SAMPLER -> matche la distribution d entree au TEST (p_test/p_train), + equilibrage genre
def sampler_weights(g,gg):
    s=importance_ratio(g)
    freq={x:(gg==x).mean() for x in (0.,1.)}
    s=s*np.array([(1.0/freq[x])**GENDER_POWER for x in gg])
    s=np.clip(s,np.percentile(s,1),min(np.percentile(s,99),OMEGA_CLIP*s.mean()))
    return (s/s.mean()).astype(np.float64)
sampler_w=sampler_weights(df_train["FaceOcclusion"].values, df_train["gender"].values)

# (2) poids de LOSS -> facteur metrique (1/30+gt), sous la distribution deja matchee par le sampler
omega_train=(1/30+df_train["FaceOcclusion"].values).astype(np.float32); omega_train/=omega_train.mean()

# verif: la distribution effective TIREE par le sampler doit coller au test
_p=sampler_w/sampler_w.sum()
_eff=df_train["FaceOcclusion"].values[np.random.RandomState(0).choice(len(sampler_w),200000,p=_p)]
print(f"sampler effective: mean={_eff.mean():.3f} >0.2={np.mean(_eff>0.2):.3f} | cible test ~0.165/0.382")
print(f"sampler_w min/med/max: {sampler_w.min():.2f}/{np.median(sampler_w):.2f}/{sampler_w.max():.2f}")

transform_train=T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)),T.RandomHorizontalFlip(),
    T.ColorJitter(0.2,0.2,0.15),T.RandomApply([T.GaussianBlur(3,(0.1,1.5))],p=0.2),
    T.RandomApply([T.RandomRotation(10)],p=0.3),T.ToTensor(),T.Normalize(MEAN,STD),
    T.RandomErasing(p=0.25,scale=(0.02,0.15))])
transform_eval=T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)),T.ToTensor(),T.Normalize(MEAN,STD)])
transform_flip=T.Compose([T.Resize((IMG_SIZE,IMG_SIZE)),T.RandomHorizontalFlip(p=1.0),T.ToTensor(),T.Normalize(MEAN,STD)])

Train 80000 Val 20000
sampler effective: mean=0.171 >0.2=0.399 | cible test ~0.165/0.382
sampler_w min/med/max: 0.24/0.45/4.42


In [4]:
class OccDataset(Dataset):
    def __init__(self,df,transform,omega=None,labeled=True):
        self.f=df["filename"].values; self.t=transform; self.labeled=labeled
        self.y=df["FaceOcclusion"].values.astype(np.float32) if labeled else None
        self.g=df["gender"].values.astype(np.float32) if labeled else None
        self.om=omega
    def __len__(self): return len(self.f)
    def __getitem__(self,i):
        img=self.t(Image.open(os.path.join(IMAGE_DIR,self.f[i])).convert("RGB"))
        if not self.labeled: return (img,)
        return img, self.y[i], self.g[i], (self.om[i] if self.om is not None else np.float32(1.0))

def make_train_loader(bs):
    ds=OccDataset(df_train,transform_train,omega=omega_train,labeled=True)
    sampler=WeightedRandomSampler(torch.as_tensor(sampler_w,dtype=torch.double),
                                  num_samples=len(sampler_w),replacement=True)  # sur-echantillonne les fortes occlusions
    return DataLoader(ds,batch_size=bs,sampler=sampler,num_workers=NUM_WORKERS,pin_memory=True,drop_last=True)
val_loader=DataLoader(OccDataset(df_val,transform_eval,labeled=True),batch_size=64,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
test_loader=DataLoader(OccDataset(df_test,transform_eval,labeled=False),batch_size=64,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)
test_loader_flip=DataLoader(OccDataset(df_test,transform_flip,labeled=False),batch_size=64,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True)

In [5]:
def build_backbone(name):
    if name=="convnext_tiny": bb=models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1); bb.classifier[2]=nn.Identity(); f=768
    elif name=="convnext_small": bb=models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1); bb.classifier[2]=nn.Identity(); f=768
    elif name=="convnext_base": bb=models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1); bb.classifier[2]=nn.Identity(); f=1024
    elif name=="swin_s": bb=models.swin_s(weights=models.Swin_S_Weights.IMAGENET1K_V1); bb.head=nn.Identity(); f=768
    else: raise ValueError(name)
    return bb,f
class Net(nn.Module):
    def __init__(self,name):
        super().__init__(); self.backbone,f=build_backbone(name)
        self.head=nn.Sequential(nn.Linear(f,256),nn.GELU(),nn.Dropout(0.3),nn.Linear(256,1),nn.Sigmoid())
    def forward(self,x): return self.head(self.backbone(x))

def omega_mse(p,gt,om): return (om*(p.squeeze(-1)-gt)**2).mean()
def fairness_term(p,gt,gd,om):
    pr=p.squeeze(-1); se=om*(pr-gt)**2; m,f=gd==1.0,gd==0.0
    if m.sum()==0 or f.sum()==0: return torch.zeros((),device=p.device)
    em=se[m].sum()/(om[m].sum()+1e-8); ef=se[f].sum()/(om[f].sum()+1e-8)
    return torch.sqrt((em-ef)**2+1e-8)

def error_fn(p,gt): w=1/30+gt; return float(np.sum(w*(p-gt)**2)/np.sum(w))
def metric_fn(p,gt,gd):
    m,f=gd==1.0,gd==0.0; em,ef=error_fn(p[m],gt[m]),error_fn(p[f],gt[f]); return (em+ef)/2+abs(em-ef),em,ef
def _err_w(p,gt,w): return float(np.sum(w*(p-gt)**2)/np.sum(w))
def metric_fn_test(p,gt,gd):
    w=(1/30+gt)*importance_ratio(gt); m,f=gd==1.0,gd==0.0
    em,ef=_err_w(p[m],gt[m],w[m]),_err_w(p[f],gt[f],w[f]); return (em+ef)/2+abs(em-ef),em,ef

class EMA:
    def __init__(self,model,decay): self.decay=decay; self.shadow={k:v.detach().clone() for k,v in model.state_dict().items()}
    @torch.no_grad()
    def update(self,model):
        for k,v in model.state_dict().items():
            if v.dtype.is_floating_point: self.shadow[k].mul_(self.decay).add_(v.detach(),alpha=1-self.decay)
            else: self.shadow[k]=v.detach().clone()

In [6]:
def infer(model,loader):
    model.eval(); out=[]
    with torch.inference_mode():
        for b in loader:
            with torch.autocast("cuda",dtype=torch.float16): out.append(model(b[0].to(device)).squeeze(-1).float().cpu().numpy())
    return np.concatenate(out)

def train_model(name):
    print(f"\n========== {name} ==========")
    bs=BATCH.get(name,32); train_loader=make_train_loader(bs)
    model=Net(name).to(device); scaler=torch.cuda.amp.GradScaler()
    opt=torch.optim.AdamW([{"params":model.backbone.parameters(),"lr":LR_BACKBONE},
                           {"params":model.head.parameters(),"lr":LR_HEAD}],weight_decay=WEIGHT_DECAY)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS,eta_min=1e-6)
    ema=EMA(model,EMA_DECAY)
    def eval_ema():
        bak={k:v.detach().clone() for k,v in model.state_dict().items()}
        model.load_state_dict(ema.shadow); pv=infer(model,val_loader); model.load_state_dict(bak); return pv
    best=float("inf"); best_sd=None
    for ep in range(1,EPOCHS+1):
        tb=ep>FREEZE_EPOCHS
        for p in model.backbone.parameters(): p.requires_grad=tb
        model.train()
        if not tb: model.backbone.eval()
        losses=[]
        for X,y,gd,om in tqdm(train_loader,desc=f"{name} ep{ep}/{EPOCHS}[{'e2e' if tb else 'head'}]"):
            X,y,gd,om=X.to(device),y.to(device),gd.to(device),om.to(device)
            opt.zero_grad()
            with torch.autocast("cuda",dtype=torch.float16):
                pred=model(X); loss=omega_mse(pred,y,om)
                if FAIRNESS_LAMBDA>0: loss=loss+FAIRNESS_LAMBDA*fairness_term(pred,y,gd,om)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
            scaler.step(opt); scaler.update(); ema.update(model); losses.append(loss.item())
        sched.step()
        pv=eval_ema()
        sc_raw,_,_=metric_fn(pv,yv,gv)
        sc,em,ef=metric_fn_test(pv,yv,gv)
        flag="  <-- best" if sc<best else ""
        print(f"  ep{ep} loss={np.mean(losses):.5f} | Score_test={sc:.5f} (M={em:.5f} F={ef:.5f}) | Score_brut={sc_raw:.5f}{flag}")
        if sc<best: best=sc; best_sd={k:v.clone() for k,v in ema.shadow.items()}
    model.load_state_dict(best_sd)
    val_pred=infer(model,val_loader)
    test_pred=(infer(model,test_loader)+infer(model,test_loader_flip))/2.0
    torch.save(best_sd,os.path.join(OUT_DIR,f"best_{name}.pt"))
    np.save(os.path.join(OUT_DIR,f"val_pred_{name}.npy"),val_pred)
    np.save(os.path.join(OUT_DIR,f"test_pred_{name}.npy"),test_pred)
    print(f"  >>> {name} meilleur Score_test val = {best:.5f}")
    del model; torch.cuda.empty_cache()
    return val_pred,test_pred,best

In [7]:
val_preds, test_preds = {}, {}
for name in BACKBONES:
    vp,tp,_=train_model(name)
    val_preds[name]=vp; test_preds[name]=tp
print("\nModeles entraines:", list(val_preds))


========== convnext_small ==========
Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /root/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth


100%|██████████| 192M/192M [00:00<00:00, 208MB/s]
/tmp/ipykernel_23/1835980856.py:11: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  model=Net(name).to(device); scaler=torch.cuda.amp.GradScaler()


convnext_small ep1/22[head]:   0%|          | 0/1666 [00:00<?, ?it/s]

  ep1 loss=0.00578 | Score_test=0.00357 (M=0.00316 F=0.00233) | Score_brut=0.00470  <-- best


convnext_small ep2/22[head]:   0%|          | 0/1666 [00:00<?, ?it/s]

...

  ep20 loss=0.00030 | Score_test=0.00160 (M=0.00125 F=0.00054) | Score_brut=0.00134


convnext_small ep21/22[e2e]:   0%|          | 0/1666 [00:00<?, ?it/s]

  ep21 loss=0.00029 | Score_test=0.00161 (M=0.00125 F=0.00054) | Score_brut=0.00134


convnext_small ep22/22[e2e]:   0%|          | 0/1666 [00:00<?, ?it/s]

  ep22 loss=0.00028 | Score_test=0.00161 (M=0.00125 F=0.00054) | Score_brut=0.00134
  >>> convnext_small meilleur Score_test val = 0.00143

Modeles entraines: ['convnext_small']


In [8]:
np.save(os.path.join(OUT_DIR,"y_val.npy"),yv); np.save(os.path.join(OUT_DIR,"g_val.npy"),gv)
for n in BACKBONES:
    sc,_,_=metric_fn_test(val_preds[n],yv,gv); print(f"  {n:16s} Score_test val = {sc:.5f}")
val_ens=np.mean([val_preds[n] for n in BACKBONES],axis=0)
test_ens=np.mean([test_preds[n] for n in BACKBONES],axis=0)
sc,em,ef=metric_fn_test(val_ens,yv,gv); print(f"\nEnsemble Score_test={sc:.5f} M={em:.5f} F={ef:.5f}")
from sklearn.isotonic import IsotonicRegression
w_iso=(1/30+yv)*importance_ratio(yv)
iso=IsotonicRegression(out_of_bounds="clip",y_min=0.0,y_max=1.0); iso.fit(val_ens,yv,sample_weight=w_iso)
sc_c,_,_=metric_fn_test(iso.predict(val_ens),yv,gv); USE_CAL=sc_c<sc
print(f"Ensemble isotone Score_test={sc_c:.5f} | appliquee:{USE_CAL}")
final=np.clip(iso.predict(test_ens) if USE_CAL else test_ens,0,1)
sub=df_test.copy(); sub["FaceOcclusion"]=final
sub.to_csv(os.path.join(OUT_DIR,"submission_distmatch.csv"),index=False)
print("\nsubmission_distmatch.csv ecrit. Compare a 0.00100, garde 0.00100 en fallback.")
print("Score_test = PROXY (val reechantillonnee). Seul le leaderboard tranche.")

  convnext_small   Score_test val = 0.00143

Ensemble Score_test=0.00143 M=0.00114 F=0.00055
Ensemble isotone Score_test=0.00129 | appliquee:True

submission_distmatch.csv ecrit. Compare a 0.00100, garde 0.00100 en fallback.
Score_test = PROXY (val reechantillonnee). Seul le leaderboard tranche.
